In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from src.preprocessing import impute_missing, convert_types
from src.features import add_features, add_neighborhood_features, add_target_encoding_features, FEATURES
from src.utils import run_cv, make_submission

In [2]:
train_df = pd.read_csv("../data/train.csv")
test_df  = pd.read_csv("../data/test.csv")

y = train_df["SalePrice"]
train_df = train_df.drop("SalePrice", axis=1)

df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

In [3]:
# 前処理
df = impute_missing(df)
df = convert_types(df)

In [4]:
# 特徴量生成
df = add_features(df)

train = df.iloc[:len(y)].copy()
test  = df.iloc[len(y):].copy()

train = add_neighborhood_features(train, train)
test  = add_neighborhood_features(test, train)

# KFold Target Encoding
train["SalePrice"] = y.values
train, test = add_target_encoding_features(train, test, target_col="SalePrice")
train = train.drop("SalePrice", axis=1)

X_train = train[FEATURES]
y_train = np.log1p(y)
X_test  = test[FEATURES]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (1460, 45)
X_test:  (1459, 45)


In [5]:
params = {
    "boosting_type" : "gbdt",
    "objective"     : "regression",
    "metric"        : "rmse",
    "learning_rate" : 0.1,
    "num_leaves"    : 16,
    "n_estimators"  : 100000,
    "random_state"  : 123,
    "importance_type": "gain",
}

metrics, imp = run_cv(X_train, y_train, params)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000703 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5133
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 45
[LightGBM] [Info] Start training from score 12.021806
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[62]	training's rmse: 0.0770986	valid_1's rmse: 0.117321
[fold 0] tr: 0.07710, va: 0.11732
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5116
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 45
[LightGBM] [Info] Start training from score 12.014343


In [6]:
imp.head(20)

,imp,imp_std
col,,
Neighborhood_TotalQual_SF,561.811492,35.048965
TotalQual_SF,204.336539,41.020448
Neighborhood_Qual,24.117484,6.785477
Neighborhood_SF,20.832520,8.281602
Garage_SF,15.880174,4.327513
OverallCond,14.806393,1.805373
TotalBath,11.144009,2.031867
GarageScore,8.893079,1.708045
BsmtFinSF1,8.467796,0.499461


In [ ]:
# 全データで再学習 → submission
model = lgb.LGBMRegressor(**params)
model.fit(X_train, y_train)

make_submission(
    model, X_test, test["Id"],
    filepath="../submissions/submission_03_feature_engineering.csv"
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000897 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5328
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 45
[LightGBM] [Info] Start training from score 12.024057
Saved: ../submissions/hp-submission_03_feature_engineering.csv


,Id,SalePrice
1460,1461,117473.119138
1461,1462,173998.007402
1462,1463,183719.487227
1463,1464,186926.156645
1464,1465,183486.177824
...,...,...
2914,2915,80720.567863
2915,2916,84775.403385
2916,2917,154945.572477
2917,2918,117609.749298
